This aim of this file is to explain which errors Beril recieved and how she handled them while tying to run the SIGLIP code of Etienne.

First a summary of what was added and changed in comparison to the original code is given. Following are the recieved errors listed chronologically, which also explain why the changes were necessary. There has been 7 important errors in total.

ADDITION means one or more new code lines are added. Most of the additions are for the setup to be complete and they don't have additional errors below.
CHANGE means a change was made to the original code.
The numbers in () next to those indicate which error caused this change/addition to be made and in which order.

The changed version of the code:

In [ ]:
# ADDITION
!pip install duckdb transformers

In [ ]:
# ADDITION
import duckdb

DB = duckdb.connect("iconclass.duckdb")
DB.execute("INSTALL vss; LOAD vss;")

In [ ]:
# CHANGE (7):

# Original: DB.execute("create table if not exists siglip (notation TEXT, vec FLOAT[512])")
# Current model outputs 1152-dimensional vectors
DB.execute("create table if not exists siglip_1152 (notation TEXT, vec FLOAT[1152])")

import torch
from transformers import AutoProcessor, AutoModel
from transformers.image_utils import load_image

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

model_id = "google/siglip2-so400m-patch16-naflex"
model = AutoModel.from_pretrained(model_id).eval()
model = model.to(device)
processor = AutoProcessor.from_pretrained(model_id)

# CHANGE (4):

# Original: BATCH_SIZE = 10000
# Reduced for memory
BATCH_SIZE = 1000

def do(batch, limit=BATCH_SIZE):
    if len(batch) < limit:
        return batch

    # CHANGE (2):
    
    # Original: text=batch.values()
    # Colab expects a list
    inputs = processor(text=list(batch.values()), padding="max_length", max_length=64, return_tensors="pt").to(model.device)

    with torch.no_grad():
        # image_embeds = model.get_image_features(pixel_values=inputs["pixel_values"])

        # CHANGE (3) and (5): (could be very wrong to do this but...)
        
        # Original:
        # text_embeddings = model.get_text_features(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        # Current output uses .pooler_output and did not use attention_mask here
        text_embeddings = model.get_text_features(input_ids=inputs["input_ids"]).pooler_output

    # image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)

    # similarity = (image_embeds @ text_embeddings.T)

    # CHANGE (6):
 
    # Original: to_insert = zip(batch.keys(), text_embeddings)
    # DuckDB needs plain Python lists, not torch tensors
    to_insert = zip(batch.keys(), text_embeddings.cpu().tolist())

    DB.executemany("INSERT INTO siglip VALUES (?, ?)", to_insert)
    DB.commit()
    return {}

In [ ]:
#ADDITION
!pip install iconclass

In [ ]:
# ADDITION
from time import time
from rich.progress import track
from iconclass import init

# ADDITION (1):

# 'all' is defined, None values are removed
IC = init()
all = [x for x in IC.source._D.keys() if x is not None]

In [ ]:
idx = 0
start_time = time()
batch = {}
LANG = 'en'
for notation in track(all):
    if notation.find('(+') > 1 and notation not in used_notation_keys:
        continue
    n = IC[notation]


    texts = {}
    batch[notation] = n(LANG)[:77]
    batch = do(batch)
    if len(batch) == 0:
        print(f"At {idx}")
    idx += 1
do(batch, limit=0)
siglip_duration = time() - start_time
print(f"SIGLIP processing duration: {siglip_duration} seconds")

Respective errors:

Error (1):  
TypeError                                 Traceback (most recent call last)  
/tmp/ipykernel_2627/3551363642.py in <cell line: 0>()  
      3 batch = {}  
      4 LANG = 'en'  
----> 5 for notation in track(all):  
      6     if notation.find('(+') > 1 and notation not in used_notation_keys:  
      7         continue  

1 frames  
/usr/local/lib/python3.12/dist-packages/rich/progress.py in track(self, sequence, total, completed, task_id, description, update_period)  
   1223         if self.live.auto_refresh:  
   1224             with _TrackThread(self, task_id, update_period) as track_thread:  
-> 1225                 for value in sequence:  
   1226                     yield value  
   1227                     track_thread.completed += 1  

TypeError: 'builtin_function_or_method' object is not iterable -> here the code tries to iterate over Python's built-in function all() and it doesn't work

Error (2): ValueError                                Traceback (most recent call last)  
/tmp/ipykernel_2627/3551363642.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  

3 frames  
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_tokenizers.py in _encode_plus(self, text, text_pair, add_special_tokens, padding_strategy, truncation_strategy, max_length, stride, is_split_into_words, pad_to_multiple_of, padding_side, return_tensors, return_token_type_ids, return_attention_mask, return_overflowing_tokens, return_special_tokens_mask, return_offsets_mapping, return_length, verbose, split_special_tokens, **kwargs)  
    900   
    901         if not _is_valid_text_input(text):  
--> 902             raise ValueError(  
    903                 "text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) "
    904                 "or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs)."  

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

Error (3): KeyError                                  Traceback (most recent call last)  
/tmp/ipykernel_2627/3551363642.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  

1 frames  
/usr/local/lib/python3.12/dist-packages/transformers/feature_extraction_utils.py in __getitem__(self, item)  
     89         """  
     90         if isinstance(item, str):  
---> 91             return self.data[item]  
     92         else:
     93             raise KeyError("Indexing with integers is not available when using Python based feature extractors")  

KeyError: 'attention_mask' -> argument feature 'attention_mask' couldn't be found so i removed it, an alternative could be better than removing


Error (4): OutOfMemoryError                          Traceback (most recent call last)  
/tmp/ipykernel_2627/3551363642.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  
 20 frames  
 /usr/local/lib/python3.12/dist-packages/torch/nn/modules/linear.py in forward(self, input)  
    132         Runs the forward pass.  
    133         """  
--> 134         return F.linear(input, self.weight, self.bias)  
    135   
    136     def extra_repr(self) -> str:  
OutOfMemoryError: CUDA out of memory. Tried to allocate 2.75 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.48 GiB is free. Including non-PyTorch memory, this process has 13.08 GiB memory in use. Of the allocated memory 12.49 GiB is allocated by PyTorch, and 479.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

Error (5): AttributeError                            Traceback (most recent call last)  
/tmp/ipykernel_2106/555002990.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  

/tmp/ipykernel_2106/1381377909.py in do(batch, limit)  
     28   
     29     # image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)  
---> 30     text_embeddings = text_embeddings / text_embeddings.norm(dim=-1, keepdim=True)  
     31   
     32     # similarity = (image_embeds @ text_embeddings.T)  

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'  

Error (6):NotImplementedException                   Traceback (most recent call last)  
/tmp/ipykernel_2106/555002990.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  

/tmp/ipykernel_2106/1965637400.py in do(batch, limit)  
     34   
     35     to_insert = zip(batch.keys(), text_embeddings)  
---> 36     DB.executemany("INSERT INTO siglip VALUES (?, ?)", to_insert)  
     37     DB.commit()  
     38     return {}  

NotImplementedException: Not implemented Error: Unable to transform python value of type '<class 'torch.Tensor'>' to DuckDB LogicalType


Error (7):ConversionException                       Traceback (most recent call last)  
/tmp/ipykernel_2106/555002990.py in <cell line: 0>()  
     11     texts = {}  
     12     batch[notation] = n(LANG)[:77]  
---> 13     batch = do(batch)  
     14     if len(batch) == 0:  
     15         print(f"At {idx}")  

/tmp/ipykernel_2106/2358474018.py in do(batch, limit)
     34 
     35     to_insert = zip(batch.keys(), text_embeddings.cpu().tolist())  
---> 36     DB.executemany("INSERT INTO siglip VALUES (?, ?)", to_insert)  
     37     DB.commit()  
     38     return {}  

ConversionException: Conversion Error: Cannot cast list with length 1152 to array with length 512